In [1]:
import numpy as np
import torch
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
from PIL import Image
from scripts.blazeface import BlazeFace, BlazeFaceWrapper
from scripts.postprocess import postprocess
from PIL import Image
from PIL import ImageDraw

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [2]:
path = "artefacts/blazeface.pth"
front_net = BlazeFace().to(device)
front_net.load_state_dict(torch.load(path))
front_net.eval()  

anchor_path = "artefacts/anchors.npy"
anchors = np.load(anchor_path)
anchors_tensor = torch.tensor(anchors).to(device).float()

model = BlazeFaceWrapper(front_net, anchors_tensor).to(device)
model.eval()
pass

In [3]:
img = Image.open("test_image.jpg").convert("RGB")
original_width, original_height = img.width, img.height
target_width, target_height = 128, 128
img_resized = img.resize((target_width, target_height))
img_tensor = torch.from_numpy(np.array(img_resized)).permute((2, 0, 1)).unsqueeze(0).to(device).float()

In [5]:
torch.onnx.export(
    model,
    (img_tensor,),
    "blaze_face.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamo=True,
)

[torch.onnx] Obtain model graph for `BlazeFaceWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BlazeFaceWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/ridwanfatur/miniconda3/envs/py3_12_9/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
Applied 20 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.13.0+cu130',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,3,128,128]>
            ),
            outputs=(
                %"output"<FLOAT,[1,896,17]>
            ),
            initializers=(
                %"blazeface.backbone1.0.weight"<FLOAT,[24,3,5,5]>{TorchTensor(...)},
                %"blazeface.backbone1.0.bias"<FLOAT,[24]>{TorchTensor(...)},
                %"blazeface.backbone1.2.convs.0.weight"<FLOAT,[24,1,3,3]>{TorchTensor(...)},
                %"blazeface.backbone1.2.convs.1.weight"<FLOAT,[24,24,1,1]>{TorchTensor(...)},
                %"blazeface.backbone1.2.convs.1.bias"<FLOAT,[24]>{TorchTensor(...)},
                %"blazeface.backbone1.3.convs.0.weight"<FLOAT,[24,1,3,3]>{TorchTensor(